# NOT training with ICNN 

In [ ]:
import os, sys
sys.path.append("..")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.models import Transport, Critic, RF_Transport, RF_Critic, ICNNCritic
from src.utils import cost, show_mapping , grad_norm, cosine_lr, plot_3d_function, ema_np, plot_loss, plot_grad_norm

In [ ]:
# source distribution μ
def sample_mu(batch_size, device="cpu"):
    x = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    y = torch.zeros(batch_size, 1, device=device)         # y = 0
    return torch.cat([x, y], dim=1)


# target distribution ν
def sample_nu(batch_size, device="cpu"):
    x = torch.zeros(batch_size, 1, device=device)         # y = 0
    y = 2 * torch.rand(batch_size, 1, device=device) - 1  # [-1,1]
    return torch.cat([x, y], dim=1)

In [ ]:
T = Transport().to(device)
f = ICNNCritic().to(device)

loss_hist = []
grad_T_hist = []
grad_f_hist = []

## GDmax Training

In [ ]:
lr_T = 1e-5
lr_f = 1e-5

K = 10

optimizer_T = torch.optim.Adam(T.parameters(), lr=lr_T)
optimizer_f = torch.optim.Adam(f.parameters(), lr=lr_f, maximize= True)

batchsize_x = 1024
batchsize_y = 1024

for step in range(20000):
    x = sample_mu(batchsize_x).to(device)
    y = sample_nu(batchsize_y).to(device)

    for _ in range(K):
        Tx = T(x)
        loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()

        optimizer_T.zero_grad()
        loss.backward()
        optimizer_T.step()

    Tx = T(x)
    loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()

    optimizer_f.zero_grad()
    loss.backward()
    optimizer_f.step()

    Tx = T(x)
    loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()

    T_grads = torch.autograd.grad(loss, T.parameters(), retain_graph=True)
    f_grads = torch.autograd.grad(loss, f.parameters())

    grad_T = grad_norm(T_grads).item()
    grad_f = grad_norm(f_grads).item()
    
    loss_hist.append(loss.item())
    grad_T_hist.append(grad_T)
    grad_f_hist.append(grad_f)
    
    if (step+1) % 500 == 0:
        lr_T = cosine_lr(step, 2000, 1e-4, 0)
        lr_f = cosine_lr(step, 2000, 1e-4, 0)
        clear_output(wait=True)
        print(f"step {step+1}, loss {loss.item():.4f}")
        show_mapping(T, sample_mu, sample_nu)

## Extragradient Training

In [ ]:
lr_T = 1e-4
lr_f = 1e-4

batchsize_x = 1024
batchsize_y = 1024
for step in range(200000):
    x = sample_mu(batchsize_x).to(device)
    y = sample_nu(batchsize_y).to(device)

    T_old = [p.clone() for p in T.parameters()]
    f_old = [p.clone() for p in f.parameters()]

    # -------- critic update (maximize) --------
    Tx = T(x)
    loss = cost(x, Tx).mean() - f(Tx).mean() + f(y).mean()
    T_gradient = torch.autograd.grad(loss, T.parameters(), create_graph=True)
    f_gradient = torch.autograd.grad(loss, f.parameters())

    with torch.no_grad():
        for p, g in zip(T.parameters(), T_gradient):
            p -= lr_T * g
        for p, g in zip(f.parameters(), f_gradient):
            p += lr_f * g
    Tx2 = T(x)
    loss2 = cost(x, Tx2).mean() - f(Tx2).mean() + f(y).mean()
    T_grad_true = torch.autograd.grad(loss2, T.parameters(), create_graph=True)
    f_grad_true = torch.autograd.grad(loss2, f.parameters())

    # ----- restore parameters -----
    with torch.no_grad():
        for p, p_old in zip(T.parameters(), T_old):
            p.copy_(p_old)
        for p, p_old in zip(f.parameters(), f_old):
            p.copy_(p_old)

    with torch.no_grad():
        for p, g in zip(T.parameters(), T_grad_true):
            p -= lr_T * g
        for p, g in zip(f.parameters(), f_grad_true):
            p += lr_f * g
    
    # To make f an ICNN, please uncomment this 
    # with torch.no_grad():   f.a.clamp_(min=0)
    
    loss_hist.append(loss.item())
    grad_T_hist.append(grad_T)
    grad_f_hist.append(grad_f)

    if (step+1) % 500 == 0:
        clear_output(wait=True)
        print(f"step {step+1}, loss {loss.item():.4f}")
        show_mapping(T, sample_mu, sample_nu, f=f, contour = True)

## Plot Results

In [ ]:
plot_loss(loss_hist)
plot_grad_norm(grad_T_hist, grad_f_hist)

In [ ]:
show_mapping(T,sample_mu,sample_nu, option = False, f=f, contour=True, legend = False)

In [ ]:
plot_3d_function(f)